In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG        = "clutchlytics"
SKATER_LOGS    = f"{CATALOG}.silver.nhl_skater_game_logs"
DIM_ATHLETES   = f"{CATALOG}.silver.dimAthletes"
GOLD_TABLE     = f"{CATALOG}.gold.nhl_gold_player_delta"
 
LEAGUE         = "nhl"
SPORT          = "hockey"
ROUND          = 1       # ← change to 2 when R2 data loads
 
# Minimum sample filters
MIN_REG_GAMES     = 10
MIN_PLAYOFF_GAMES = 2
 
# Flags thresholds
ELEVATED_PTS_DELTA    =  0.3   # pts/game above regular season
DISAPPEARED_PTS_DELTA = -0.4   # pts/game below regular season
PRODUCTIVE_REG_THRESHOLD = 0.4 # min reg pts/game to be flagged as disappeared
ROLE_EXPANDED_TOI_DELTA  = 180 # seconds (~3 min) above regular season avg
ELEVATED_TOI_DELTA       =  60 # seconds (~1 min) above regular season avg
 
print(f"Source       : {SKATER_LOGS}")
print(f"Target       : {GOLD_TABLE}")
print(f"Round        : {ROUND}")
print(f"Min reg games: {MIN_REG_GAMES}")
print(f"Min po games : {MIN_PLAYOFF_GAMES}")

In [0]:
# ── READ SOURCES ──────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
 
skater_df = spark.table(SKATER_LOGS)
 
dim_athletes = (
    spark.table(DIM_ATHLETES)
    .filter(F.col("is_current") == True)
    .select(
        F.col("athlete_id").cast("string").alias("dim_athlete_id"),
        F.col("clutch_athlete_id"),
        F.col("clutch_team_id"),
        F.col("position_abbr"),
        F.col("position_group"),
    )
)
 
print(f"Skater log rows : {skater_df.count()}")
print(f"dimAthletes     : {dim_athletes.count()}")

In [0]:
# ── REGULAR SEASON BASELINE ───────────────────────────────────────────────────
# Aggregate regular season stats per player — 2026 season only.
 
reg_df = (
    skater_df
    .filter(F.col("season_type") == "regular")
    .groupBy("athlete_id", "athlete_name", "team_abbreviation")
    .agg(
        F.count("*").alias("reg_games"),
        F.round(F.avg("points"), 3).alias("reg_pts_per_game"),
        F.round(F.avg("goals"), 3).alias("reg_goals_per_game"),
        F.round(F.avg("assists"), 3).alias("reg_assists_per_game"),
        F.round(F.avg("shots"), 3).alias("reg_shots_per_game"),
        F.round(F.avg("toi_seconds"), 1).alias("reg_toi_avg_seconds"),
        F.round(F.avg("plus_minus"), 3).alias("reg_plus_minus_avg"),
    )
    .filter(F.col("reg_games") >= MIN_REG_GAMES)
)
 
print(f"Players with reg season baseline (>= {MIN_REG_GAMES} games): {reg_df.count()}")
 



In [0]:
# ── PLAYOFF PERFORMANCE ───────────────────────────────────────────────────────
# Aggregate playoff stats for the target round.
 
playoff_df = (
    skater_df
    .filter(
        (F.col("season_type") == "playoffs") &
        (F.col("round") == ROUND)
    )
    .groupBy("athlete_id")
    .agg(
        F.count("*").alias("playoff_games"),
        F.round(F.avg("points"), 3).alias("playoff_pts_per_game"),
        F.round(F.avg("goals"), 3).alias("playoff_goals_per_game"),
        F.round(F.avg("assists"), 3).alias("playoff_assists_per_game"),
        F.round(F.avg("shots"), 3).alias("playoff_shots_per_game"),
        F.round(F.avg("toi_seconds"), 1).alias("playoff_toi_avg_seconds"),
        F.round(F.avg("plus_minus"), 3).alias("playoff_plus_minus_avg"),
        F.sum("goals").alias("playoff_total_goals"),
        F.sum("points").alias("playoff_total_points"),
    )
    .filter(F.col("playoff_games") >= MIN_PLAYOFF_GAMES)
)
 
print(f"Players with playoff data (round {ROUND}, >= {MIN_PLAYOFF_GAMES} games): {playoff_df.count()}")

In [0]:
# ── JOIN + COMPUTE DELTAS ─────────────────────────────────────────────────────
# Inner join — excludes players missing either baseline or playoff data.
# Players with zero playoff games already excluded by playoff_df filter.
 
delta_df = (
    reg_df
    .join(playoff_df, on="athlete_id", how="inner")
    .join(
        dim_athletes,
        reg_df.athlete_id == dim_athletes.dim_athlete_id,
        how="left"
    )
    .drop("dim_athlete_id")
)
 
# ── Raw deltas ──
delta_df = (
    delta_df
    .withColumn("pts_delta",
        F.round(F.col("playoff_pts_per_game") - F.col("reg_pts_per_game"), 3))
    .withColumn("goals_delta",
        F.round(F.col("playoff_goals_per_game") - F.col("reg_goals_per_game"), 3))
    .withColumn("assists_delta",
        F.round(F.col("playoff_assists_per_game") - F.col("reg_assists_per_game"), 3))
    .withColumn("shots_delta",
        F.round(F.col("playoff_shots_per_game") - F.col("reg_shots_per_game"), 3))
    .withColumn("toi_delta_seconds",
        F.round(F.col("playoff_toi_avg_seconds") - F.col("reg_toi_avg_seconds"), 1))
    .withColumn("plus_minus_delta",
        F.round(F.col("playoff_plus_minus_avg") - F.col("reg_plus_minus_avg"), 3))
)
 
# ── TOI delta rank within position group ──
# Uses ESPN position groups as-is: Centers, Defense, Left Wings, Right Wings
# More precise than F/D split — center TOI expectations differ from wingers
pos_group_window = Window.partitionBy("position_group").orderBy(
    F.col("toi_delta_seconds").desc()
)
 
delta_df = delta_df.withColumn(
    "toi_delta_rank_in_position",
    F.rank().over(pos_group_window)
)
 
# ── Position group size for percentile context ──
pos_group_size = (
    delta_df
    .groupBy("position_group")
    .agg(F.count("*").alias("position_group_size"))
)
 
delta_df = delta_df.join(pos_group_size, on="position_group", how="left")
 
delta_df = delta_df.withColumn(
    "toi_delta_pct_in_position",
    F.round(
        (1 - (F.col("toi_delta_rank_in_position") - 1) /
         F.col("position_group_size")) * 100, 1
    )
)

In [0]:
# ── FLAGS ─────────────────────────────────────────────────────────────────────
 
delta_df = (
    delta_df
    # Elevated — producing significantly more AND getting more ice time
    .withColumn(
        "elevated_flag",
        (F.col("pts_delta") > ELEVATED_PTS_DELTA) &
        (F.col("toi_delta_seconds") > ELEVATED_TOI_DELTA)
    )
    # Role expanded — significantly more ice time regardless of points
    .withColumn(
        "role_expanded_flag",
        F.col("toi_delta_seconds") > ROLE_EXPANDED_TOI_DELTA
    )
    # Disappeared — producing significantly less AND was productive in reg season
    .withColumn(
        "disappeared_flag",
        (F.col("pts_delta") < DISAPPEARED_PTS_DELTA) &
        (F.col("reg_pts_per_game") > PRODUCTIVE_REG_THRESHOLD)
    )
)

In [0]:
# ── FINAL COLUMN ORDER + ADD ROUND ───────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
gold_df = (
    delta_df
    .select(
        # ── Identity ──
        F.col("athlete_id"),
        F.col("athlete_name"),
        F.col("clutch_athlete_id"),
        F.col("team_abbreviation"),
        F.col("position_abbr"),
        F.col("position_group"),
 
        # ── Round context ──
        F.lit(ROUND).alias("round"),
        F.lit(2026).alias("season"),
        F.lit(SPORT).alias("sport"),
        F.lit(LEAGUE).alias("league"),
 
        # ── Regular season baseline ──
        F.col("reg_games"),
        F.col("reg_pts_per_game"),
        F.col("reg_goals_per_game"),
        F.col("reg_assists_per_game"),
        F.col("reg_shots_per_game"),
        F.col("reg_toi_avg_seconds"),
        F.col("reg_plus_minus_avg"),
 
        # ── Playoff performance ──
        F.col("playoff_games"),
        F.col("playoff_pts_per_game"),
        F.col("playoff_goals_per_game"),
        F.col("playoff_assists_per_game"),
        F.col("playoff_shots_per_game"),
        F.col("playoff_toi_avg_seconds"),
        F.col("playoff_plus_minus_avg"),
        F.col("playoff_total_goals"),
        F.col("playoff_total_points"),
 
        # ── Deltas ──
        F.col("pts_delta"),
        F.col("goals_delta"),
        F.col("assists_delta"),
        F.col("shots_delta"),
        F.col("toi_delta_seconds"),
        F.col("plus_minus_delta"),
 
        # ── TOI context within position group ──
        F.col("toi_delta_rank_in_position"),
        F.col("position_group_size"),
        F.col("toi_delta_pct_in_position"),
 
        # ── Flags ──
        F.col("elevated_flag"),
        F.col("role_expanded_flag"),
        F.col("disappeared_flag"),
 
        # ── Metadata ──
        F.lit(ingested_at).alias("ingested_at"),
        F.lit("silver.nhl_skater_game_logs").alias("source_table"),
    )
    .orderBy("position_group", F.col("pts_delta").desc())
)
 
print(f"Total rows to write: {gold_df.count()}")

In [0]:
# ── PREVIEW ───────────────────────────────────────────────────────────────────
 
print(f"── Top 15 elevated players (Round {ROUND}) ──")
gold_df.filter(F.col("elevated_flag") == True).select(
    "athlete_name", "team_abbreviation", "position_group",
    "reg_pts_per_game", "playoff_pts_per_game", "pts_delta",
    "reg_toi_avg_seconds", "playoff_toi_avg_seconds", "toi_delta_seconds",
    "toi_delta_pct_in_position"
).orderBy(F.col("pts_delta").desc()).show(15, truncate=False)
 
print(f"\n── Top 10 disappeared players (Round {ROUND}) ──")
gold_df.filter(F.col("disappeared_flag") == True).select(
    "athlete_name", "team_abbreviation", "position_group",
    "reg_pts_per_game", "playoff_pts_per_game", "pts_delta",
    "toi_delta_seconds"
).orderBy(F.col("pts_delta")).show(10, truncate=False)
 
print(f"\n── Flag summary ──")
gold_df.agg(
    F.sum(F.col("elevated_flag").cast("int")).alias("elevated_count"),
    F.sum(F.col("role_expanded_flag").cast("int")).alias("role_expanded_count"),
    F.sum(F.col("disappeared_flag").cast("int")).alias("disappeared_count"),
).show(truncate=False)

In [0]:
# ── WRITE TO GOLD ─────────────────────────────────────────────────────────────
# MERGE on athlete_id + round — preserves R1 rows when R2 is added later.
 
table_exists = spark.catalog.tableExists(GOLD_TABLE)
 
if not table_exists:
    (
        gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_TABLE)
    )
    print(f"Table created: {GOLD_TABLE}")
else:
    gold_df.createOrReplaceTempView("new_player_delta")
    spark.sql(f"""
        MERGE INTO {GOLD_TABLE} AS target
        USING new_player_delta AS source
        ON  target.athlete_id = source.athlete_id
        AND target.round      = source.round
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table: {GOLD_TABLE}")

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_players,
        COUNT(DISTINCT position_group)                              AS position_groups,
        COUNT(CASE WHEN elevated_flag   = true THEN 1 END)         AS elevated,
        COUNT(CASE WHEN role_expanded_flag = true THEN 1 END)      AS role_expanded,
        COUNT(CASE WHEN disappeared_flag = true THEN 1 END)        AS disappeared,
        COUNT(CASE WHEN pts_delta > 0 THEN 1 END)                  AS positive_delta,
        COUNT(CASE WHEN pts_delta < 0 THEN 1 END)                  AS negative_delta,
        COUNT(CASE WHEN pts_delta = 0 THEN 1 END)                  AS zero_delta,
        ROUND(AVG(pts_delta), 3)                                    AS avg_pts_delta,
        ROUND(AVG(toi_delta_seconds), 1)                           AS avg_toi_delta,
        COUNT(CASE WHEN clutch_athlete_id IS NULL THEN 1 END)      AS null_clutch_ids
    FROM {GOLD_TABLE}
    WHERE round = {ROUND}
""")
 
print(f"Sanity checks (Round {ROUND}):")
checks.show(truncate=False)